# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows in mock mode (approving only 1 and 2, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.engine import run_request
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    session = run_request(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, session.status)

chf_safe_haven_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


buy_the_dip_tech completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

230 run records, 45 counted by the page, 2795 events


total                      230.000000
completed                   94.000000
blocked                     41.000000
planner_refused             47.000000
failed                       0.000000
rejected                     3.000000
completed_share              0.408696
blocked_share                0.178261
planner_refused_share        0.204348
failed_share                 0.000000
rejected_share               0.013043
completed_percent           41.000000
blocked_percent             18.000000
planner_refused_percent     20.000000
failed_percent               0.000000
rejected_percent             1.000000
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

`activity` splits the runs into five outcomes that do not overlap (completed, blocked
by a deterministic guardrail, refused by the planner with P01, failed, rejected by the
reviewer), so they add up to the total; `_percent` holds the whole percentages the
page shows, which add up to 100.

In [4]:
from quantlab.monitoring import (
    completed_with_warnings,
    grounded_narratives,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **completed_with_warnings(records),
    }
)

approved               91.000000
rejected                3.000000
approval_rate           0.968085
edited                  7.000000
edit_rate               0.076923
approval_median_s       0.033000
approval_p95_s         76.798500
replays                12.000000
matching               12.000000
replay_match_rate       1.000000
narratives             94.000000
grounded               94.000000
grounded_rate           1.000000
completed              94.000000
with_warnings          31.000000
with_warnings_share     0.329787
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,459
1,V09,LOOK_AHEAD,APPROVE,60
2,P01,PLANNER_REFUSED,BLOCK,47
3,E04,ALIGNMENT_LOSS,WARN,46
4,V03,INVALID_PARAM,BLOCK,32
5,V08,FREQUENCY_MISMATCH,BLOCK,31
6,V02,UNKNOWN_CAPABILITY,BLOCK,19
7,R01,REQUEST_TOO_LONG,BLOCK,1
8,V01,INVALID_PLAN,BLOCK,0
9,V04,TYPE_MISMATCH,BLOCK,0


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,127
1,resample,C02,109
2,flag_regime,C07,90
3,forward_return,C08,70
4,hac_mean_difference,C10,70
5,compare_groups,C09,54
6,compute_returns,C03,49
7,rolling_volatility,C06,25
8,spread,C05,10
9,difference,C04,0


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,47,3775.765957,5329.8,388.702128,897.1,0.000572,0.000906,0.026881
1,narrator,47,505.340426,1645.9,63.404255,186.0,0.000082,0.000253,0.003865
2,total,47,4281.106383,6520.3,452.106383,1038.2,0.000654,0.001116,0.030746


planner_calls             47.000000
cache_hits                 9.000000
hit_rate                   0.191489
saved_input_tokens     42023.000000
saved_output_tokens     5517.000000
saved_cost                 0.006961
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,229,1020.578603,7980.3200,True
2,Validator,229,0.145459,0.3600,True
3,Executor,94,122.937021,189.1145,True
4,Narrator,94,583.105319,3053.3600,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-19T10:38:42.374Z,20261019-103842-3e433e,blocked,Validator,V02,The plan has blocking issues: V02:s2
1,2026-10-19T10:38:42.348Z,20261019-103842-61a0c9,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-19T10:38:42.320Z,20261019-103842-8389c6,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-19T10:28:41.731Z,20261019-102841-b0f427,blocked,Validator,V08,The plan has blocking issues: V08:s4
4,2026-10-19T10:16:12.904Z,20261019-101612-9ba0c7,blocked,Validator,V02,The plan has blocking issues: V02:s2
